# Chapter 3: LLMs for Text Classification and Generation
**Module 04: Introduction to LLMs in Python**

> Source integrated from `chapter3.pdf`.

## Learning Objectives
- Compare `pipeline()` with Hugging Face Auto classes.
- Load pretrained models for classification, generation, summarization, translation, and QA.
- Explore datasets used for supervised LLM tasks.
- Explain fine-tuning and transfer learning workflows.

## Chapter Map
| Section | Focus |
|---|---|
| 3.1 | Loading pretrained LLMs |
| 3.2 | Auto classes for classification and generation |
| 3.3 | Datasets for classification and generation |
| 3.4 | Summarization and translation |
| 3.5 | Question answering |
| 3.6 | Fine-tuning and transfer learning |


## 3.1 Loading a Pretrained LLM

| Approach | Strength | Tradeoff |
|---|---|---|
| `pipeline()` | Simple high-level interface with automatic model/tokenizer selection | Less control and less flexibility |
| `AutoModel` and task Auto classes | More customization and fine-tuning support | More manual setup |

> **Tip:** Use `pipeline()` for quick experiments. Use Auto classes when you need hidden states, logits, custom heads, or training control.


## 3.2 `AutoModel` and `AutoTokenizer`

`from_pretrained()` loads model weights and tokenizer configuration from a checkpoint. `AutoModel` returns the model body without a task-specific head, so this section adds a simple classifier head manually.


In [ ]:
# If needed, install dependencies first:
# pip install transformers datasets torch sentencepiece accelerate

import torch
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
text = "I am an example sequence for text classification."

class SimpleClassifier(nn.Module):
    def __init__(self, input_size, num_classes):
        super().__init__()
        self.fc = nn.Linear(input_size, num_classes)

    def forward(self, x):
        return self.fc(x)


In [ ]:
inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=64)
outputs = model(**inputs)
pooled_output = outputs.pooler_output

print("Hidden states size:", outputs.last_hidden_state.shape)
print("Pooled output size:", pooled_output.shape)

classifier_head = SimpleClassifier(pooled_output.size(-1), num_classes=2)
logits = classifier_head(pooled_output)
probs = torch.softmax(logits, dim=1)
print("Predicted Class Probabilities:", probs)


## 3.3 Auto Class for Text Classification

`AutoModelForSequenceClassification` includes a classification head, so logits are available directly.


In [ ]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model_name = "nlptown/bert-base-multilingual-uncased-sentiment"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

text = "The quality of the product was just okay."
inputs = tokenizer(text, return_tensors="pt")
outputs = model(**inputs)
logits = outputs.logits
predicted_class = torch.argmax(logits, dim=1).item()
print(f"Predicted class index: {predicted_class + 1} star.")


## 3.4 Auto Class for Text Generation

`AutoModelForCausalLM` is configured for autoregressive next-token prediction. `generate()` repeatedly predicts the next token and appends it to the sequence.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

prompt = "This is a simple example for text generation,"
inputs = tokenizer.encode(prompt, return_tensors="pt")
output = model.generate(inputs, max_length=26)
generated_text = tokenizer.decode(output[0], skip_special_tokens=True)

print("Generated Text:")
print(generated_text)


## 3.5 Exploring Datasets

The `datasets` library loads task datasets from the Hugging Face hub. Classification examples pair text with labels; generation datasets often contain prompts, histories, or target responses.


In [ ]:
from datasets import load_dataset
from torch.utils.data import DataLoader

dataset = load_dataset("imdb")
train_data = dataset["train"]
dataloader = DataLoader(train_data, batch_size=2, shuffle=True)

batch = next(iter(dataloader))
for i in range(len(batch["text"])):
    print(f"Example {i + 1}:")
    print("Text:", batch["text"][i][:300], "...")
    print("Label:", batch["label"][i])


In [ ]:
from datasets import load_dataset

dataset = load_dataset("stanfordnlp/shp", "askculinary")
train_data = dataset["train"]

for i in range(5):
    example = train_data[i]
    print(f"Example {i + 1}:")
    print("Title:", example["post_id"])
    print("Paragraph:", example["history"][:300], "...")
    print()


## 3.6 How Text Generation Training Works

Text generation uses shifted input-target pairs.

| Source Text | Input Sequence | Target Sequence |
|---|---|---|
| `the cat is sleeping on the mat` | `the cat is` | `cat is sleeping` |

The target is shifted one token to the left, teaching the model to predict the next token.


## 3.7 Summarization

| Type | Description |
|---|---|
| Extractive summarization | Selects and combines parts of the original text |
| Abstractive summarization | Generates a new summary word by word |

The PDF uses the ILSUM English dataset and `t5-small` for sequence-to-sequence summarization.


In [ ]:
from datasets import load_dataset

dataset = load_dataset("ILSUM/ILSUM-1.0", "English")
print(f"Features: {dataset['train'].column_names}")

example = dataset["train"][21]
print("Article preview:", example["Article"][:500], "...")
print("Summary:", example["Summary"])


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

input_ids = tokenizer.encode("summarize: " + example["Article"], return_tensors="pt", max_length=512, truncation=True)
summary_ids = model.generate(input_ids, max_length=150)
summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("Original Text:")
print(example["Article"][:1000], "...")
print("\nGenerated Summary:")
print(summary)


## 3.8 Translation

Translation models encode a source-language sequence and decode the corresponding target-language sequence. The PDF demonstrates English-to-Welsh translation.


In [ ]:
from datasets import load_dataset

dataset = load_dataset("techiaith/legislation-gov-uk_en-cy")
sample_data = dataset["train"]
input_example = sample_data.data["source"][0].as_py()
target_example = sample_data.data["target"][0].as_py()

print("Input (English):", input_example)
print("Target (Welsh):", target_example)


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "Helsinki-NLP/opus-mt-en-cy"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

input_seq = "2 Regulations under section 1: supplementary"
input_ids = tokenizer.encode(input_seq, return_tensors="pt")
translated_ids = model.generate(input_ids)
translated_text = tokenizer.decode(translated_ids[0], skip_special_tokens=True)
print("Predicted (Welsh):", translated_text)


## 3.9 Question Answering

| QA Type | Architecture | Behavior |
|---|---|---|
| Extractive | Encoder-only | Extracts an answer span from context |
| Open generative | Encoder-decoder | Generates an answer from context |
| Closed generative | Decoder-only | Generates an answer without supplied context |


In [ ]:
from datasets import load_dataset

mlqa = load_dataset("xtreme", name="MLQA.en.en")
print(mlqa)
print("Question:", mlqa["test"]["question"][53])
print("Answer:", mlqa["test"]["answers"][53])
print("Context:", mlqa["test"]["context"][53][:700], "...")


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

model_ckp = "deepset/minilm-uncased-squad2"
tokenizer = AutoTokenizer.from_pretrained(model_ckp)
model = AutoModelForQuestionAnswering.from_pretrained(model_ckp)

question = "How is the taste of wasabi?"
context = """Japanese cuisine captures the essence of a harmonious fusion between fresh ingredients and
traditional culinary techniques, all heightened by the zesty taste of the aromatic green condiment known as wasabi."""
inputs = tokenizer(question, context, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

start_idx = torch.argmax(outputs.start_logits)
end_idx = torch.argmax(outputs.end_logits) + 1
answer_span = inputs["input_ids"][0][start_idx:end_idx]
answer = tokenizer.decode(answer_span)
print(answer)


In [ ]:
example_qt = mlqa["test"]["question"][53]
example_ct = mlqa["test"]["context"][53]
long_exmp = tokenizer(example_qt, example_ct, return_overflowing_tokens=True, max_length=100, stride=25)

for idx, window in enumerate(long_exmp["input_ids"]):
    print("Tokens in window", idx, ":", len(window))

for window in long_exmp["input_ids"][:2]:
    print(tokenizer.decode(window), "\n")


## 3.10 Fine-Tuning and Transfer Learning

| Technique | What Changes | Cost |
|---|---|---|
| Full fine-tuning | All model weights update | Higher compute |
| Partial fine-tuning | Lower/body layers fixed; selected layers or head update | Lower compute |
| Zero-shot | No task-specific training examples | No training cost |
| One-shot / few-shot | One or a few examples guide adaptation | Low data cost |

Transfer learning adapts a model trained on one task to a related task, usually with less data than training from scratch.


In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

data = load_dataset("imdb")
tokenized_datasets = data.map(tokenize_function, batched=True)


In [ ]:
# Optional fine-tuning cell. This can take time and benefits from a GPU.
from transformers import Trainer, TrainingArguments

small_train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(128))
small_eval_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(128))

training_args = TrainingArguments(
    output_dir="./smaller_bert_finetuned",
    per_device_train_batch_size=8,
    num_train_epochs=1,
    eval_strategy="steps",
    eval_steps=50,
    save_steps=50,
    logging_dir="./logs",
    report_to="none",
)

trainer = Trainer(model=model, args=training_args, train_dataset=small_train_dataset, eval_dataset=small_eval_dataset)

# Uncomment when you are ready to run fine-tuning.
# trainer.train()


In [ ]:
example_input = tokenizer("I am absolutely amazed with this new and revolutionary AI device", return_tensors="pt")
output = model(**example_input)
predicted_label = torch.argmax(output.logits, dim=1).item()
print("Predicted Label:", predicted_label)

# Save after fine-tuning.
# model.save_pretrained("./my_bert_finetuned")
# tokenizer.save_pretrained("./my_bert_finetuned")


## Chapter Summary
- `pipeline()` is ideal for quick demos; Auto classes are better for control and fine-tuning.
- Classification heads expose logits; causal language models generate with `generate()`.
- Datasets define the input-target structure for each task.
- Summarization and translation are sequence-to-sequence tasks.
- Extractive QA predicts answer start and end token positions.
- Fine-tuning adapts pretrained LLMs to downstream tasks.
